# 对称性应用 完整教程：从复杂枚举到优雅简化

## 📚 教学目标
1. 理解**对称性 (Symmetry)** 在简化问题中的核心作用
2. 掌握**硬币分堆**问题中对称性的精妙运用
3. 学会**贴错标签**问题的逻辑推理与对称性论证
4. 理解**对称博弈**中先手/后手优势的判断方法
5. 使用 Python 模拟验证对称性策略的正确性

**参考书**：《A Practical Guide to Quantitative Finance Interviews》(Xinfeng Zhou) 第2章
**教学策略**：先用小规模例子手算建立直觉，再用 Python 模拟/枚举验证分析结果

---

## 1. 为什么对称性如此重要？

### 🎯 核心思想

许多看似复杂的问题，一旦发现其中的**对称结构**，就能大幅简化求解过程。

对称性的应用模式：

```
复杂问题 → 发现对称结构 → 利用对称性降维 → 优雅的简单解
```

在量化面试中，对称性思维是区分「暴力求解」和「优雅解题」的关键。面试官往往不只是要答案，更要看你能否发现问题中隐藏的结构。

### 💡 对称性的三种常见形式

| 类型 | 描述 | 典型例子 |
|------|------|----------|
| 数值对称 | 某种操作使不同状态等价 | 硬币翻转 |
| 信息对称 | 利用标签与内容的全错配 | 贴错标签问题 |
| 策略对称 | 镜像/模仿对手操作 | 圆桌硬币博弈 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import permutations, product

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 硬币分堆问题 (Coin Piles Problem)

### 🎯 问题描述

桌上有 **100 枚硬币**，其中 **20 枚正面朝上**，80 枚反面朝上。房间完全漆黑，你看不到也摸不出硬币的正反面。

**要求**：将硬币分成两堆，使得两堆中**正面朝上的硬币数量相同**。

### 📐 初步分析

看起来不可能？你完全看不到硬币状态，怎么可能让两堆正面数相同？

暴力思路：随机分堆，成功概率极低。我们需要找到一种**确定性策略**。

In [ ]:
# ========== 步骤 1: 暴力方法 —— 随机分堆有多难？ ==========
np.random.seed(42)

N_COINS = 100
N_HEADS = 20
N_SIMULATIONS = 100000

# 模拟随机分堆
success_count = 0
for _ in range(N_SIMULATIONS):
    # 创建硬币: 1=正面, 0=反面
    coins = np.array([1]*N_HEADS + [0]*(N_COINS - N_HEADS))
    np.random.shuffle(coins)
    
    # 随机选择分堆大小 (1 到 99)
    split = np.random.randint(1, N_COINS)
    pile1 = coins[:split]
    pile2 = coins[split:]
    
    if pile1.sum() == pile2.sum():
        success_count += 1

print(f"📊 步骤 1: 随机分堆的成功率")
print(f"  模拟次数: {N_SIMULATIONS:,}")
print(f"  成功次数: {success_count:,}")
print(f"  成功率: {success_count/N_SIMULATIONS*100:.2f}%")
print(f"")
print(f"  💡 随机分堆几乎不可能成功！我们需要更聪明的策略。")

### 📐 对称性解法

**关键洞察**：取出任意 20 枚硬币作为一堆，然后**将这 20 枚全部翻转**。

为什么有效？设取出的 20 枚中有 $k$ 枚正面朝上：

| 堆 | 硬币数 | 正面朝上数 |
|---|---|---|
| 取出的 20 枚（翻转前） | 20 | $k$ |
| 剩余的 80 枚 | 80 | $20 - k$ |
| 取出的 20 枚（翻转后） | 20 | $20 - k$ |

翻转后，取出堆的正面数 = $20 - k$，恰好等于剩余堆的正面数！

$$\text{翻转后正面数} = 20 - k = \text{剩余堆正面数}$$

### 💡 对称性在哪里？

无论 $k$ 取什么值（0 到 20），翻转操作都能将取出堆的正面数变为 $20-k$。
这个策略对**所有可能的初始配置**都有效 —— 这就是对称性的力量。

In [ ]:
# ========== 步骤 2: 手算验证 —— 小规模例子 ==========
print("📊 步骤 2: 手算验证 (10 枚硬币, 4 枚正面)")
print("─" * 60)

n_small = 10
h_small = 4  # 正面朝上数
pick = h_small  # 取出 4 枚

print(f"  设定: {n_small} 枚硬币, {h_small} 枚正面朝上")
print(f"  策略: 取出 {pick} 枚, 全部翻转")
print()

for k in range(pick + 1):
    heads_picked = k
    heads_remaining = h_small - k
    heads_after_flip = pick - k
    match = "✅" if heads_after_flip == heads_remaining else "❌"
    
    print(f"  k={k}: 取出堆有 {heads_picked} 正面 → 翻转后 {heads_after_flip} 正面 | "
          f"剩余堆 {heads_remaining} 正面 {match}")

print(f"\n  💡 无论 k 是多少, 翻转后两堆正面数永远相等！")
print(f"     这就是 '取出 N_heads 枚 + 全部翻转' 策略的数学保证。")

In [ ]:
# ========== 步骤 3: Python 模拟验证 —— 100 枚硬币 ==========
np.random.seed(42)

N_COINS = 100
N_HEADS = 20
N_SIMULATIONS = 50000

success_count = 0
k_values = []  # 记录每次取出堆中的正面数

for _ in range(N_SIMULATIONS):
    # 创建随机排列的硬币
    coins = np.array([1]*N_HEADS + [0]*(N_COINS - N_HEADS))
    np.random.shuffle(coins)
    
    # 策略: 取出前 20 枚 (位置无所谓, 因为已经随机排列)
    pile_picked = coins[:N_HEADS].copy()
    pile_remaining = coins[N_HEADS:]
    
    k = pile_picked.sum()  # 取出堆中的正面数
    k_values.append(k)
    
    # 翻转取出堆: 1->0, 0->1
    pile_picked_flipped = 1 - pile_picked
    
    # 检查两堆正面数是否相等
    if pile_picked_flipped.sum() == pile_remaining.sum():
        success_count += 1

print(f"📊 步骤 3: 对称性策略的模拟验证")
print(f"  模拟次数: {N_SIMULATIONS:,}")
print(f"  成功次数: {success_count:,}")
print(f"  成功率: {success_count/N_SIMULATIONS*100:.2f}%")
print(f"")
print(f"  💡 成功率 = 100%！策略对所有随机配置都有效！")
print(f"")
print(f"  取出堆中正面数 k 的分布:")
k_arr = np.array(k_values)
for k_val in sorted(set(k_values)):
    count = (k_arr == k_val).sum()
    if count > 100:
        print(f"    k={k_val:2d}: {count:5d} 次 ({count/N_SIMULATIONS*100:.1f}%)")

In [ ]:
# ========== 可视化: k 的分布 (超几何分布) ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: k 的实际分布 ---
ax1 = axes[0]
k_unique, k_counts = np.unique(k_values, return_counts=True)
k_freq = k_counts / N_SIMULATIONS

bars = ax1.bar(k_unique, k_freq, color='steelblue', edgecolor='black', alpha=0.85)
ax1.set_xlabel('k (Heads in Picked Pile)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of k in Picked Pile\n(Hypergeometric Distribution)', 
              fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 理论超几何分布叠加
k_theory = np.arange(0, N_HEADS + 1)
pmf_theory = stats.hypergeom.pmf(k_theory, N_COINS, N_HEADS, N_HEADS)
ax1.plot(k_theory, pmf_theory, 'r-o', linewidth=2, markersize=5, 
         label='Hypergeometric PMF', zorder=5)
ax1.legend(fontsize=10)

# --- 右图: 策略演示图 ---
ax2 = axes[1]
k_demo = np.arange(0, N_HEADS + 1)
heads_flipped = N_HEADS - k_demo  # 翻转后正面数
heads_remain = N_HEADS - k_demo   # 剩余堆正面数

ax2.plot(k_demo, heads_flipped, 'o-', color='#2ecc71', linewidth=2.5, 
         markersize=8, label='Flipped Pile Heads', zorder=5)
ax2.plot(k_demo, heads_remain, 's--', color='#e74c3c', linewidth=2.5, 
         markersize=8, label='Remaining Pile Heads', zorder=4)

ax2.set_xlabel('k (Initial Heads in Picked Pile)', fontsize=12)
ax2.set_ylabel('Number of Heads', fontsize=12)
ax2.set_title('Strategy Verification: Both Piles Always Match', 
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

textstr = 'Flipped heads = 20 - k\nRemaining heads = 20 - k\nAlways equal!'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax2.text(0.98, 0.98, textstr, transform=ax2.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：取出 20 枚中正面数 k 服从超几何分布, 最可能的值是 k=4")
print(f"  右图：无论 k 取何值, 翻转后两堆的正面数完全重合 (绿线和红线重叠)")
print(f"        这证明了策略的普适性 —— 对所有 k 值都有效")

---

## 3. 贴错标签的水果袋 (Mislabeled Bags)

### 🎯 问题描述

有 3 个袋子，分别装着：
- 袋子 A：只有**苹果**
- 袋子 B：只有**橙子**
- 袋子 C：**苹果和橙子的混合**

每个袋子上都贴了标签，但**所有标签都贴错了**。

**要求**：只从一个袋子中取出一个水果，就能确定所有袋子的正确内容。

### 📐 初步分析

3 个袋子有 $3! = 6$ 种排列方式，但「所有标签都贴错」这一约束会大大缩小可能性。

贴错标签在数学上称为**错排 (Derangement)**：每个元素都不在原来的位置上。

In [ ]:
# ========== 步骤 4: 枚举所有错排 ==========
print("📊 步骤 4: 枚举所有可能的错排")
print("─" * 60)

contents = ['Apples', 'Oranges', 'Mixed']
labels = ['Apples', 'Oranges', 'Mixed']  # 标签（都是错的）

print(f"  标签顺序 (固定): {labels}")
print(f"  实际内容的所有排列:")
print()

all_perms = list(permutations(contents))
derangements = []

for perm in all_perms:
    is_derangement = all(perm[i] != labels[i] for i in range(3))
    status = "✅ 错排" if is_derangement else "❌ 不是错排"
    matches = sum(1 for i in range(3) if perm[i] == labels[i])
    
    print(f"  {perm} → 匹配数={matches} → {status}")
    if is_derangement:
        derangements.append(perm)

print(f"\n  总排列数: {len(all_perms)}")
print(f"  错排数: {len(derangements)}")
print(f"\n  所有错排:")
for d in derangements:
    print(f"    袋子[标签苹果]={d[0]}, 袋子[标签橙子]={d[1]}, 袋子[标签混合]={d[2]}")

print(f"\n  💡 只有 {len(derangements)} 种可能! 远少于 6 种全排列。")

### 📐 对称性解法

**关键洞察**：从**标签为「混合」的袋子**中取出一个水果。

为什么？因为：
1. 标签「混合」是错的 → 这个袋子里**不是**混合的
2. 所以这个袋子要么全是苹果，要么全是橙子
3. 取出一个就能确定这个袋子的真实内容

确定一个袋子后，利用「所有标签都贴错」的约束，其余两个也随之确定：

$$\text{1 个确定} \xrightarrow{\text{错排约束}} \text{3 个全部确定}$$

### 💡 为什么从其他袋子取不行？

如果从标签「苹果」的袋子取出一个苹果——这个袋子可能是「混合」（混合袋也有苹果），无法确定。

In [ ]:
# ========== 步骤 5: 逻辑推理验证 ==========
print("📊 步骤 5: 从标签'混合'袋取水果的推理")
print("─" * 60)

print("\n  场景 1: 从标签'混合'袋取出 → 苹果")
print("  ─────────────────────────────────")
print("  • 标签'混合'的袋子实际是: 苹果 (因为标签错了, 不是混合)")
print("  • 标签'苹果'的袋子不能是苹果(标签错) → 只能是橙子或混合")
print("  • 标签'橙子'的袋子不能是橙子(标签错) → 只能是苹果或混合")
print("  • 苹果已被占用 → 标签'橙子'的袋子是: 混合")
print("  • 剩下 → 标签'苹果'的袋子是: 橙子")
print("  ✅ 结果: [标签苹果]=橙子, [标签橙子]=混合, [标签混合]=苹果")

print("\n  场景 2: 从标签'混合'袋取出 → 橙子")
print("  ─────────────────────────────────")
print("  • 标签'混合'的袋子实际是: 橙子")
print("  • 标签'橙子'的袋子不能是橙子(标签错) → 只能是苹果或混合")
print("  • 标签'苹果'的袋子不能是苹果(标签错) → 只能是橙子或混合")
print("  • 橙子已被占用 → 标签'苹果'的袋子是: 混合")
print("  • 剩下 → 标签'橙子'的袋子是: 苹果")
print("  ✅ 结果: [标签苹果]=混合, [标签橙子]=苹果, [标签混合]=橙子")

print("\n  💡 两种场景都恰好对应一种错排! 取一个水果即可完全确定。")

In [ ]:
# ========== 步骤 6: 对比 —— 从其他袋子取会怎样？ ==========
print("📊 步骤 6: 为什么从其他袋子取不行?")
print("─" * 60)

print("\n  从标签'苹果'的袋子取出水果:")
print("  该袋子实际可能是: 橙子 或 混合 (两种错排都满足)")
print()

for d in derangements:
    actual_in_apple_label = d[0]  # 标签苹果的袋子里实际是什么
    print(f"  错排 {d}:")
    print(f"    标签'苹果'袋里实际是: {actual_in_apple_label}")
    if actual_in_apple_label == 'Oranges':
        print(f"    取出 → 橙子 → 确定是橙子袋 ✅")
    elif actual_in_apple_label == 'Mixed':
        print(f"    取出 → 可能苹果也可能橙子 → 不确定! ❌")
        print(f"    如果取出苹果: 无法区分这是'只有苹果'还是'混合'")

print(f"\n  💡 从标签'苹果'袋取出苹果时, 无法区分是纯苹果袋还是混合袋")
print(f"     信息量不足以确定 → 策略失败!")
print(f"")
print(f"  关键区别:")
print(f"    标签'混合'袋 → 实际只能是纯苹果或纯橙子 → 取一个就确定")
print(f"    标签'苹果'袋 → 实际可能是橙子或混合 → 取苹果时有歧义")

In [ ]:
# ========== 步骤 7: Monte Carlo 模拟验证 ==========
np.random.seed(42)
N_SIM = 100000

# 模拟策略: 从标签'混合'的袋子取一个
correct_from_mixed = 0
correct_from_apple = 0

for _ in range(N_SIM):
    # 随机选一个错排
    d = derangements[np.random.randint(len(derangements))]
    # d[0]=标签苹果实际, d[1]=标签橙子实际, d[2]=标签混合实际
    
    # --- 策略 A: 从标签'混合'取 ---
    actual_mixed_label = d[2]  # 标签混合的袋子实际内容
    if actual_mixed_label == 'Apples':
        drawn = 'apple'  # 纯苹果袋, 一定取出苹果
    else:  # Oranges
        drawn = 'orange'  # 纯橙子袋, 一定取出橙子
    
    # 根据取出结果推断
    if drawn == 'apple':
        guess = ('Oranges', 'Mixed', 'Apples')
    else:
        guess = ('Mixed', 'Apples', 'Oranges')
    
    if guess == d:
        correct_from_mixed += 1
    
    # --- 策略 B: 从标签'苹果'取 ---
    actual_apple_label = d[0]
    if actual_apple_label == 'Oranges':
        drawn_b = 'orange'
    elif actual_apple_label == 'Mixed':
        drawn_b = 'apple' if np.random.random() < 0.5 else 'orange'
    else:
        drawn_b = 'apple'
    
    # 尝试推断 (最优猜测)
    if drawn_b == 'orange' and actual_apple_label == 'Oranges':
        # 可能确定标签苹果袋是橙子
        # 但 drawn_b == 'orange' 也可能来自 Mixed 袋
        pass
    # 简化: 假设取到橙子就猜是纯橙子袋, 取到苹果就猜是混合袋
    if drawn_b == 'orange':
        guess_b = ('Oranges', 'Mixed', 'Apples')
    else:
        guess_b = ('Mixed', 'Apples', 'Oranges')
    
    if guess_b == d:
        correct_from_apple += 1

print(f"📊 步骤 7: 两种策略的 Monte Carlo 对比")
print(f"  模拟次数: {N_SIM:,}")
print(f"")
print(f"  策略 A (从标签'混合'取): 正确率 = {correct_from_mixed/N_SIM*100:.1f}%")
print(f"  策略 B (从标签'苹果'取): 正确率 = {correct_from_apple/N_SIM*100:.1f}%")
print(f"")
print(f"  💡 策略 A 的正确率是 100%!")
print(f"     策略 B 只有约 50%, 因为从混合袋取出的水果有歧义。")

In [ ]:
# ========== 可视化: 错排与策略选择 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 两种策略正确率对比 ---
ax1 = axes[0]
strategies = ['Draw from\n"Mixed" label', 'Draw from\n"Apples" label']
accuracies = [correct_from_mixed/N_SIM*100, correct_from_apple/N_SIM*100]
colors = ['#2ecc71', '#e74c3c']

bars = ax1.bar(strategies, accuracies, color=colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, v in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=14)

ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Strategy Comparison: Which Bag to Draw From?', 
              fontsize=14, fontweight='bold')
ax1.set_ylim(0, 115)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: 错排数占比 (n=3 的情况) ---
ax2 = axes[1]
# 计算 n=2,3,4,5,6 的错排数
def count_derangements(n):
    if n == 0: return 1
    if n == 1: return 0
    d = [0] * (n + 1)
    d[0] = 1
    d[1] = 0
    for i in range(2, n + 1):
        d[i] = (i - 1) * (d[i-1] + d[i-2])
    return d[n]

ns = range(2, 9)
derange_counts = [count_derangements(n) for n in ns]
factorials = [np.math.factorial(n) for n in ns]
ratios = [d/f for d, f in zip(derange_counts, factorials)]

ax2.bar(ns, ratios, color='steelblue', edgecolor='black', alpha=0.85)
ax2.axhline(y=1/np.e, color='#e74c3c', linestyle='--', linewidth=2, 
            label=f'1/e = {1/np.e:.4f}')

for n_val, r in zip(ns, ratios):
    ax2.text(n_val, r + 0.005, f'{r:.3f}', ha='center', va='bottom', fontsize=10)

ax2.set_xlabel('n (Number of Items)', fontsize=12)
ax2.set_ylabel('D(n) / n!', fontsize=12)
ax2.set_title('Derangement Ratio D(n)/n! Converges to 1/e', 
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：从标签'混合'袋取的策略正确率 100%, 从其他袋取只有 ~50%")
print(f"  右图：错排数占全排列的比例随 n 增大迅速收敛到 1/e ≈ 0.368")
print(f"        对于 n=3, 错排只有 2 种(占 6 种排列的 1/3)")

---

## 4. 对称博弈：圆桌硬币游戏 (Coin on Round Table)

### 🎯 问题描述

两个玩家轮流在一张**圆形桌子**上放硬币（相同大小）。硬币不能重叠，不能超出桌面。无法放置硬币的玩家输。

**问题**：先手有必胜策略吗？

### 📐 分析

暴力分析：桌子上的放置位置几乎是连续的，似乎无法枚举。但对称性提供了优雅的解法。

**关键洞察**：先手将第一枚硬币放在桌子的**正中心**，之后采用**镜像策略** —— 后手放哪里，先手就放关于中心的对称位置。

$$\text{先手放中心} \rightarrow \text{后手放位置 P} \rightarrow \text{先手放位置 P 的关于中心对称点 P'}$$

### 💡 为什么镜像策略必胜？

1. 只要后手能放，先手的镜像位置也一定能放（因为对称性，如果 P 合法，P' 也合法）
2. 因此先手永远不会先无法放置
3. 最终是后手先无法放置 → 先手必胜

In [ ]:
# ========== 步骤 8: 圆桌硬币游戏模拟 ==========
np.random.seed(42)

def simulate_coin_game(table_radius, coin_radius, strategy='mirror', n_games=1000):
    """
    模拟圆桌硬币游戏
    
    Parameters
    ----------
    table_radius : 桌子半径
    coin_radius : 硬币半径
    strategy : 先手策略 ('mirror' 或 'random')
    n_games : 模拟局数
    
    Returns
    -------
    first_player_wins : 先手获胜次数
    """
    first_wins = 0
    
    for _ in range(n_games):
        placed_coins = []  # 已放置硬币的圆心坐标
        
        def can_place(x, y):
            """检查 (x,y) 处能否放置硬币"""
            # 硬币不能超出桌面
            if np.sqrt(x**2 + y**2) + coin_radius > table_radius:
                return False
            # 不能与已放置的硬币重叠
            for cx, cy in placed_coins:
                if np.sqrt((x-cx)**2 + (y-cy)**2) < 2 * coin_radius:
                    return False
            return True
        
        def find_random_placement(max_attempts=200):
            """随机找一个合法位置"""
            for _ in range(max_attempts):
                angle = np.random.uniform(0, 2*np.pi)
                r = np.random.uniform(0, table_radius - coin_radius)
                x, y = r * np.cos(angle), r * np.sin(angle)
                if can_place(x, y):
                    return x, y
            return None, None
        
        game_over = False
        current_player = 1  # 1 = 先手, 2 = 后手
        
        # 先手第一步: 放在中心
        if strategy == 'mirror':
            placed_coins.append((0, 0))
            current_player = 2
        
        while not game_over:
            if current_player == 1 and strategy == 'mirror' and len(placed_coins) > 1:
                # 先手使用镜像策略: 放在后手上一步的对称位置
                last_x, last_y = placed_coins[-1]
                mirror_x, mirror_y = -last_x, -last_y
                if can_place(mirror_x, mirror_y):
                    placed_coins.append((mirror_x, mirror_y))
                else:
                    # 镜像位置不可用, 随机放
                    x, y = find_random_placement()
                    if x is None:
                        game_over = True
                        break
                    placed_coins.append((x, y))
            else:
                # 后手 或 先手随机策略
                x, y = find_random_placement()
                if x is None:
                    game_over = True
                    break
                placed_coins.append((x, y))
            
            current_player = 3 - current_player  # 切换玩家
        
        # game_over 时 current_player 无法放置 → 另一方获胜
        winner = 3 - current_player
        if winner == 1:
            first_wins += 1
    
    return first_wins

# 参数设置
TABLE_R = 10
COIN_R = 1
N_GAMES = 2000

print(f"📊 步骤 8: 圆桌硬币游戏 Monte Carlo 模拟")
print(f"  桌子半径: {TABLE_R}, 硬币半径: {COIN_R}")
print(f"  模拟局数: {N_GAMES}")
print()

# 镜像策略
wins_mirror = simulate_coin_game(TABLE_R, COIN_R, strategy='mirror', n_games=N_GAMES)
print(f"  先手使用镜像策略: 胜率 = {wins_mirror/N_GAMES*100:.1f}%")

# 随机策略
wins_random = simulate_coin_game(TABLE_R, COIN_R, strategy='random', n_games=N_GAMES)
print(f"  先手使用随机策略: 胜率 = {wins_random/N_GAMES*100:.1f}%")

print(f"\n  💡 镜像策略的胜率显著高于随机策略!")
print(f"     理论上镜像策略是必胜的, 模拟中因为随机搜索的有限尝试可能略低于 100%。")

In [ ]:
# ========== 可视化: 一局游戏的放置过程 ==========
np.random.seed(123)

# 模拟一局完整的游戏并记录过程
TABLE_R = 10
COIN_R = 1.2
placed_coins_vis = [(0, 0)]  # 先手放中心
players_vis = [1]

def can_place_vis(x, y):
    if np.sqrt(x**2 + y**2) + COIN_R > TABLE_R:
        return False
    for cx, cy in placed_coins_vis:
        if np.sqrt((x-cx)**2 + (y-cy)**2) < 2 * COIN_R:
            return False
    return True

# 模拟若干步
for step in range(12):
    current = 2 if step % 2 == 0 else 1
    
    if current == 2:  # 后手随机放
        placed = False
        for _ in range(500):
            angle = np.random.uniform(0, 2*np.pi)
            r = np.random.uniform(0, TABLE_R - COIN_R)
            x, y = r * np.cos(angle), r * np.sin(angle)
            if can_place_vis(x, y):
                placed_coins_vis.append((x, y))
                players_vis.append(2)
                placed = True
                break
        if not placed:
            break
    else:  # 先手镜像
        last_x, last_y = placed_coins_vis[-1]
        mx, my = -last_x, -last_y
        if can_place_vis(mx, my):
            placed_coins_vis.append((mx, my))
            players_vis.append(1)
        else:
            break

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# 画桌子
circle_table = plt.Circle((0, 0), TABLE_R, fill=False, color='black', linewidth=2)
ax.add_patch(circle_table)

# 画硬币
for i, ((x, y), player) in enumerate(zip(placed_coins_vis, players_vis)):
    color = '#2ecc71' if player == 1 else '#e74c3c'
    label = 'Player 1 (Mirror)' if player == 1 and i <= 1 else ('Player 2 (Random)' if player == 2 and i <= 2 else None)
    coin = plt.Circle((x, y), COIN_R, color=color, alpha=0.6, edgecolor='black', linewidth=1)
    ax.add_patch(coin)
    ax.text(x, y, str(i+1), ha='center', va='center', fontsize=8, fontweight='bold')

# 画对称线
for i in range(len(placed_coins_vis)):
    if players_vis[i] == 2 and i + 1 < len(placed_coins_vis) and players_vis[i+1] == 1:
        x1, y1 = placed_coins_vis[i]
        x2, y2 = placed_coins_vis[i+1]
        ax.plot([x1, x2], [y1, y2], 'k--', alpha=0.3, linewidth=1)

ax.set_xlim(-TABLE_R - 1, TABLE_R + 1)
ax.set_ylim(-TABLE_R - 1, TABLE_R + 1)
ax.set_aspect('equal')
ax.set_title('Coin on Round Table: Mirror Strategy Demo', fontsize=14, fontweight='bold')

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', alpha=0.6, label='Player 1 (Mirror)'),
    Patch(facecolor='#e74c3c', edgecolor='black', alpha=0.6, label='Player 2 (Random)')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色 = 先手 (Player 1), 红色 = 后手 (Player 2)")
print(f"  数字 = 放置顺序。先手第 1 步放在中心")
print(f"  后续每步, 先手都在后手的中心对称位置放硬币")
print(f"  只要后手能放, 先手的镜像位置也一定能放 → 先手必胜")

---

## 5. 帽子问题中的对称性 (Symmetry in Hat Problems)

### 🎯 问题描述

3 个人各戴一顶帽子，帽子颜色为红或蓝（各有一定概率，独立随机）。每个人可以看到另外两个人的帽子，但看不到自己的。

**规则**：
- 每个人可以选择「猜自己帽子颜色」或「不猜」
- 如果所有猜的人都猜对 → 团队获胜
- 如果任何一个人猜错 → 团队失败
- 所有人都不猜 → 团队失败

**问题**：团队事先商量策略，最大获胜概率是多少？

### 📐 分析

如果每个人独立随机猜，成功率 = 50%。但利用**对称性和相关性**，可以做到 75%！

**最优策略**：
- 如果看到另外两人帽子颜色**相同** → **不猜**
- 如果看到另外两人帽子颜色**不同** → 猜自己的帽子与多数颜色**相反**

更精确地说：如果看到两顶相同颜色的帽子，猜自己的帽子是另一种颜色；如果看到两顶不同颜色的帽子，不猜。

In [ ]:
# ========== 步骤 9: 枚举所有帽子组合 ==========
print("📊 步骤 9: 帽子问题 —— 枚举所有情况")
print("─" * 60)

# R=红=0, B=蓝=1
color_map = {0: 'R', 1: 'B'}

# 所有可能的 3 人帽子组合
all_combos = list(product([0, 1], repeat=3))

print(f"  所有 2^3 = {len(all_combos)} 种帽子组合:")
print()

def optimal_strategy(hats, player_idx):
    """
    最优策略: 看到两顶相同的帽子 → 猜相反颜色; 看到两顶不同 → 不猜
    
    Returns: (action, guess)
        action: 'guess' or 'pass'
        guess: 0 or 1 (only valid if action='guess')
    """
    others = [hats[j] for j in range(3) if j != player_idx]
    if others[0] == others[1]:
        # 看到两顶相同 → 猜相反
        return 'guess', 1 - others[0]
    else:
        # 看到两顶不同 → 不猜
        return 'pass', None

wins = 0
losses = 0
details = []

for hats in all_combos:
    actions = []
    all_correct = True
    any_guess = False
    
    for p in range(3):
        action, guess = optimal_strategy(hats, p)
        actions.append((action, guess))
        if action == 'guess':
            any_guess = True
            if guess != hats[p]:
                all_correct = False
    
    result = 'WIN' if (any_guess and all_correct) else 'LOSE'
    if result == 'WIN':
        wins += 1
    else:
        losses += 1
    
    hat_str = ''.join(color_map[h] for h in hats)
    action_strs = []
    for p, (act, guess) in enumerate(actions):
        if act == 'guess':
            correct = '✅' if guess == hats[p] else '❌'
            action_strs.append(f"P{p+1}:guess {color_map[guess]}{correct}")
        else:
            action_strs.append(f"P{p+1}:pass")
    
    emoji = '🏆' if result == 'WIN' else '💀'
    print(f"  帽子={hat_str} | {', '.join(action_strs)} | {emoji} {result}")

print(f"\n  获胜: {wins}/{len(all_combos)} = {wins/len(all_combos)*100:.1f}%")
print(f"  失败: {losses}/{len(all_combos)} = {losses/len(all_combos)*100:.1f}%")
print(f"\n  💡 最优策略的获胜概率 = 75%! 远高于随机猜测的 50%。")
print(f"     策略在全同色 (RRR, BBB) 时失败, 其余 6 种情况全部获胜。")

In [ ]:
# ========== 步骤 10: Monte Carlo 验证不同策略 ==========
np.random.seed(42)
N_SIM = 100000

def simulate_hat_game(strategy_fn, n_sim):
    wins = 0
    for _ in range(n_sim):
        hats = tuple(np.random.randint(0, 2, 3))
        actions = [strategy_fn(hats, p) for p in range(3)]
        
        any_guess = False
        all_correct = True
        for p, (action, guess) in enumerate(actions):
            if action == 'guess':
                any_guess = True
                if guess != hats[p]:
                    all_correct = False
        
        if any_guess and all_correct:
            wins += 1
    return wins / n_sim

# 策略 1: 最优策略
def strategy_optimal(hats, p):
    return optimal_strategy(hats, p)

# 策略 2: 所有人都随机猜
def strategy_all_random(hats, p):
    return 'guess', np.random.randint(0, 2)

# 策略 3: 只有一个人猜 (随机)
def strategy_one_guess(hats, p):
    if p == 0:
        return 'guess', np.random.randint(0, 2)
    return 'pass', None

# 策略 4: 反向 —— 看到相同猜相同
def strategy_naive(hats, p):
    others = [hats[j] for j in range(3) if j != p]
    if others[0] == others[1]:
        return 'guess', others[0]  # 猜相同 (而非相反)
    return 'pass', None

print(f"📊 步骤 10: 四种策略的 Monte Carlo 对比")
print(f"  模拟次数: {N_SIM:,}")
print(f"─" * 50)

strategies = [
    ('最优策略 (看到相同猜相反)', strategy_optimal),
    ('反向策略 (看到相同猜相同)', strategy_naive),
    ('所有人随机猜', strategy_all_random),
    ('只有一人随机猜', strategy_one_guess),
]

strategy_results = []
for name, fn in strategies:
    win_rate = simulate_hat_game(fn, N_SIM)
    strategy_results.append((name, win_rate))
    print(f"  {name}: {win_rate*100:.1f}%")

print(f"\n  💡 最优策略 (75%) 几乎是'所有人随机猜'的 3 倍!")
print(f"     '看到相同猜相同'只有 25% —— 恰好与最优策略互补。")

In [ ]:
# ========== 可视化: 帽子策略对比 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 策略胜率对比 ---
ax1 = axes[0]
names = [s[0] for s in strategy_results]
rates = [s[1]*100 for s in strategy_results]
colors = ['#2ecc71', '#e74c3c', '#e67e22', 'steelblue']

bars = ax1.barh(range(len(names)), rates, color=colors, edgecolor='black', alpha=0.85)
for bar, r in zip(bars, rates):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{r:.1f}%', va='center', fontweight='bold', fontsize=12)

ax1.set_yticks(range(len(names)))
ax1.set_yticklabels([n[:15] + '...' if len(n) > 15 else n for n in names], fontsize=10)
ax1.set_xlabel('Win Rate (%)', fontsize=12)
ax1.set_title('Hat Problem: Strategy Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 100)
ax1.grid(axis='x', alpha=0.3)

# --- 右图: 8 种组合的胜负分布 ---
ax2 = axes[1]
combo_labels = [''.join(color_map[h] for h in c) for c in all_combos]
combo_results = []
for hats in all_combos:
    actions = [optimal_strategy(hats, p) for p in range(3)]
    any_guess = any(a[0] == 'guess' for a in actions)
    all_correct = all(a[1] == hats[p] for p, a in enumerate(actions) if a[0] == 'guess')
    combo_results.append(1 if (any_guess and all_correct) else 0)

bar_colors = ['#2ecc71' if r == 1 else '#e74c3c' for r in combo_results]
ax2.bar(range(len(combo_labels)), combo_results, color=bar_colors, 
        edgecolor='black', alpha=0.85)
ax2.set_xticks(range(len(combo_labels)))
ax2.set_xticklabels(combo_labels, fontsize=11)
ax2.set_ylabel('Win (1) / Lose (0)', fontsize=12)
ax2.set_title('Optimal Strategy: Win/Lose per Combination', fontsize=14, fontweight='bold')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Lose', 'Win'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：最优策略 (75%) 远超其他策略")
print(f"  右图：最优策略只在全同色 (RRR, BBB) 时失败, 其余 6 种全部获胜")
print(f"        对称性: 全同色正好是 2/8 = 25% 的情况")

---

## 6. 对称性思维总结：方法论

### 📐 对称性应用的一般框架

```
步骤 1: 识别问题中的对称结构
    ↓
步骤 2: 利用对称性构造不变量
    ↓
步骤 3: 用不变量简化求解
    ↓
步骤 4: 用暴力枚举/模拟验证
```

### 📐 三个问题的对称性类比

| 问题 | 对称结构 | 不变量 | 策略 |
|------|----------|--------|------|
| 硬币分堆 | 翻转操作的互补性 | 翻转后正面数 = 20-k | 取 20 枚全翻 |
| 贴错标签 | 错排约束 | 标签混合袋一定不是混合 | 从混合标签袋取 |
| 圆桌博弈 | 圆的中心对称 | 镜像位置始终合法 | 先占中心再镜像 |
| 帽子问题 | 同色/异色的互补 | P(全同色) = 25% | 见同猜反 |

In [ ]:
# ========== 综合汇总报告 ==========
print("=" * 60)
print("📋 对称性应用综合报告")
print("=" * 60)

print(f"\n🎯 核心思想:")
print(f"   对称性是化繁为简的利器。发现问题中的对称结构,")
print(f"   往往能将看似不可能的问题转化为优雅的简单解。")

print(f"\n📊 四个经典问题:")
print(f"   {'问题':<20} {'对称类型':<15} {'关键策略':<25} {'验证结果'}")
print(f"   {'─'*20} {'─'*15} {'─'*25} {'─'*12}")
print(f"   {'硬币分堆':<20} {'数值对称':<15} {'取N枚全翻转':<25} {'100%成功'}")
print(f"   {'贴错标签':<20} {'信息对称':<15} {'从混合标签袋取':<25} {'100%确定'}")
print(f"   {'圆桌博弈':<20} {'空间对称':<15} {'占中心+镜像':<25} {'先手必胜'}")
print(f"   {'帽子问题':<20} {'概率对称':<15} {'见同猜反':<25} {'75%获胜'}")

print(f"\n🧮 面试技巧:")
print(f"   1. 遇到复杂问题, 先问自己: 有没有对称性可以利用?")
print(f"   2. 对称性往往隐藏在'翻转/镜像/互补'操作中")
print(f"   3. 发现对称性后, 构造利用对称性的策略")
print(f"   4. 用暴力枚举或 Monte Carlo 验证策略的正确性")

print("\n" + "=" * 60)

---

## 7. 核心概念回顾

### 📌 对称性 (Symmetry)
- **定义**: 问题中某种变换下保持不变的结构性质
- **应用**: 通过发现对称结构，将复杂问题降维到简单问题
- **判断标准**: 寻找翻转、镜像、旋转、互补等操作下的不变量

### 📌 错排 (Derangement)
- **定义**: 所有元素都不在原位置的排列
- **公式**: $D(n) = n! \sum_{k=0}^{n} \frac{(-1)^k}{k!}$
- **含义**: 错排数占全排列的比例趋近于 $1/e \approx 36.8\%$
- **判断标准**: $D(3) = 2$, $D(4) = 9$, $D(5) = 44$

### 📌 镜像策略 (Mirror Strategy)
- **定义**: 模仿对手的操作，在对称位置执行相同动作
- **应用**: 对称博弈中，先手占据对称中心后采用镜像策略
- **含义**: 只要对手的操作合法，我的镜像操作也一定合法

### 📌 信息不对称利用
- **定义**: 利用已知约束(如「所有标签都错」)来最大化一次观察的信息量
- **应用**: 选择能产生最大信息增益的观察点
- **含义**: 贴错标签问题中，从「混合」标签袋取能一步确定所有

### 🔗 完整流程
```
分析问题结构 → 识别对称性 → 构造利用对称性的策略 → 验证策略
      ↓              ↓               ↓                ↓
  枚举所有情况    翻转/镜像/互补    确定性策略     Python模拟
```

### 📝 问题类型汇总

| 问题类型 | 对称性形式 | 解题关键 | 验证方法 |
|---------|-----------|---------|--------|
| 分配问题 | 操作互补性 | 找到保持不变量的操作 | 枚举所有 k 值 |
| 标签问题 | 错排约束 | 选信息增益最大的观察点 | 列举所有错排 |
| 博弈问题 | 空间对称 | 先占对称中心 | Monte Carlo |
| 概率问题 | 条件互补 | 利用条件概率的对称分布 | 枚举+模拟 |

---

## 8. 常见误区

### ❌ 误区 1: 硬币分堆时取出的枚数可以随意选择
**✓ 正确理解**: 必须恰好取出与正面朝上总数相同的枚数（本例中 20 枚）。取其他数量无法保证翻转后正面数相等。关键公式是「翻转后正面数 = 取出数 - k」，只有取出数 = 总正面数时，才能保证等于「总正面数 - k」。

### ❌ 误区 2: 贴错标签问题中从任何袋子取都可以
**✓ 正确理解**: 必须从标签为「混合」的袋子取。因为只有这个袋子的实际内容一定是纯的（纯苹果或纯橙子），取一个就能确定。从其他袋子取出的水果可能来自混合袋，产生歧义。

### ❌ 误区 3: 圆桌博弈中先手随便放第一枚就行
**✓ 正确理解**: 先手必须把第一枚放在桌子正中心。只有中心是唯一的中心对称点，后续才能对每个位置找到唯一的镜像位置。如果第一枚不在中心，镜像策略无法实施。

### ❌ 误区 4: 帽子问题中每个人独立决策，不需要团队策略
**✓ 正确理解**: 团队策略的关键在于「关联性」—— 利用看到的信息来决定猜或不猜。独立随机猜测的成功率（~12.5% 如果所有人都猜）远低于协调策略的 75%。对称性体现在策略对所有人一视同仁。

### ❌ 误区 5: 对称性只是一种碰巧的技巧，不具有一般性
**✓ 正确理解**: 对称性是深层数学结构的体现，广泛存在于组合、概率、博弈论中。识别对称性是一种可以系统训练的思维方式：遇到复杂问题时，先寻找翻转、镜像、互补、旋转等操作下的不变量。